# Setup

In [ ]:
import os

import numpy as np
import pandas as pd

import plotly.express as px
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt

from kneebow.rotor import Rotor
from tqdm.notebook import tqdm, trange

plt.rcParams["figure.dpi"] = 200
sns.set_palette("deep")
sns.set_context("paper")
sns.set_style("whitegrid")

In [ ]:
summary = pd.read_csv('../../data/genome_summary_Oct_12_23', sep='\t', dtype='object')
metadata = pd.read_csv('../../data/genome_metadata_Oct_12_23', sep='\t', dtype='object')

summary.shape

# Filter for species of interest


__ESKAPEEs:__
- _Enterococcus faecium_
- _Staphylococcus aureus_
- _Klebsiella pneumoniae_
- _Acinetobacter baumannii_
- _Pseudomonas aeruginosa_
- _Enterobacter spp_ (genus-level analysis)
- _Escherichia coli_ (including _Shigella_)


__Other pathogens of interest:__
- _Pseudomonas putida_
- _Pseudomonas syringae_
- _Mycobacterium tuberculosis_
- _Salmonella_ (genus-level analysis)


__Other species:__
- _Bacillus subtilis_
- _Streptococcus pyogenes_

In [ ]:
# Main filration workflow methods

def filter_by_species(summary, SPECIES_NAME):
    species_summary = summary[summary["genome_name"].str.contains(SPECIES_NAME)] # Filter for only SPECIES_NAME strains 
    species_summary = species_summary.dropna(subset=['genome_length']) # must have reported genome_length
    species_summary = species_summary.dropna(subset=['patric_cds']) # must have reported patric_cds
    
    # Ensure genome_length and patric_cds are ints
    species_summary['genome_length'] = species_summary.genome_length.astype('int')
    species_summary['patric_cds'] = species_summary.patric_cds.astype('int')

    return species_summary


def filter_by_genome_quality(
    species_summary,
    min_thresh_n50=None,
    max_contig=None,
    contamination_cutoff=None,
    completeness_cutoff=None,
    return_stats=True,
):
    
    # Split genomes into "Complete" & "WGS" bins
    species_complete_summary = species_summary[species_summary.genome_status == 'Complete']
    species_wgs_summary = species_summary[species_summary.genome_status == 'WGS']

    # internal function (for filtration)
    def get_length():
        return species_complete_summary.shape[0] + species_wgs_summary.shape[0]
    
    # Record initial lengths (for filtration statistics)
    filtration_metrics_list = ['prefiltration', 'L50/N50', 'contig_count', 'CheckM_completeness_contamination']
    filtration_columns = ['initial', 'num_filtered', 'remaining']
    df_filtration = pd.DataFrame(index=filtration_metrics_list, columns=filtration_columns)
    
    df_filtration.loc['prefiltration', 'initial'] = species_summary.shape[0]
    df_filtration.loc['prefiltration', 'remaining'] = df_filtration.loc['prefiltration', 'initial']
    df_filtration.loc['prefiltration', 'num_filtered'] = \
        df_filtration.loc['prefiltration', 'initial'] \
        - df_filtration.loc['prefiltration', 'remaining']
    
    # Filter complete sequences by L50 & N50 score metrics
    species_complete_summary = _filter_l50(species_complete_summary)
    species_complete_summary = _filter_n50(species_complete_summary, min_thresh_n50)
    
    # Record L50 & N50 filration metrics
    df_filtration.loc['L50/N50', 'initial'] = df_filtration.loc['prefiltration', 'remaining']
    df_filtration.loc['L50/N50', 'remaining'] = species_complete_summary.shape[0] + species_wgs_summary.shape[0]
    df_filtration.loc['L50/N50', 'num_filtered'] = \
        df_filtration.loc['L50/N50', 'initial'] \
        - df_filtration.loc['L50/N50', 'remaining']
    
    # Filter other WGS sequences by contig count
    species_wgs_summary = _filter_by_contig(species_wgs_summary, max_contig)
    
    # Record contig count filtration metrics
    df_filtration.loc['contig_count', 'initial'] = df_filtration.loc['L50/N50', 'remaining']
    df_filtration.loc['contig_count', 'remaining'] = species_complete_summary.shape[0] + species_wgs_summary.shape[0]
    df_filtration.loc['contig_count', 'num_filtered'] = \
        df_filtration.loc['contig_count', 'initial'] \
        - df_filtration.loc['contig_count', 'remaining']
    
    # Further filter other WGS sequences by CheckM contamination & completeness score metrics
    species_wgs_summary = _filter_checkM_contamination(species_wgs_summary, contamination_cutoff)
    species_wgs_summary = _filter_checkM_completeness(species_wgs_summary, completeness_cutoff)

    df_filtration.loc['CheckM_completeness_contamination', 'initial'] = df_filtration.loc['contig_count', 'remaining']
    df_filtration.loc['CheckM_completeness_contamination', 'remaining'] = species_complete_summary.shape[0] + species_wgs_summary.shape[0]
    df_filtration.loc['CheckM_completeness_contamination', 'num_filtered'] = \
        df_filtration.loc['CheckM_completeness_contamination', 'initial'] \
        - df_filtration.loc['CheckM_completeness_contamination', 'remaining']
    
    # Merge complete sequences and WGS sequences metadata
    filtered_species_summary = pd.concat([species_complete_summary, species_wgs_summary])

    # Typecase relevant columns as numeric
    filtered_species_summary['contig_l50'] = filtered_species_summary['contig_l50'].astype('int')
    filtered_species_summary['contig_n50'] = filtered_species_summary['contig_n50'].astype('int')
    filtered_species_summary['contigs'] = filtered_species_summary['contigs'].astype('int')
    filtered_species_summary['checkm_contamination'] = filtered_species_summary['checkm_contamination'].astype('float')
    filtered_species_summary['checkm_completeness'] = filtered_species_summary['checkm_completeness'].astype('float')

    filtered_species_summary['gc_content'] = filtered_species_summary['gc_content'].astype('float')
    
    if return_stats:
        return filtered_species_summary, df_filtration
    else:
        return filtered_species_summary

# Individual filtration functions

def _filter_l50(species_complete_summary, l50_score=1):
    species_complete_summary = species_complete_summary.dropna(subset=['contig_l50']) # must have reported L50 score
    species_complete_summary.loc[:,'contig_l50'] = species_complete_summary['contig_l50'].astype('int') # typecase to numeric
    
    good_l50 = species_complete_summary.contig_l50 == l50_score # Must be 1 for (most) bacterial complete seqs
    species_complete_summary = species_complete_summary[good_l50]

    return species_complete_summary


def _filter_n50(species_complete_summary, min_thresh_n50):
    species_complete_summary = species_complete_summary.dropna(subset=['contig_n50']) # must have reported N50 score
    species_complete_summary['contig_n50'] = species_complete_summary['contig_n50'].astype('int') # typecase to numeric
    
    if min_thresh_n50:
        cond = species_complete_summary.contig_n50 > min_thresh_n50
        species_complete_summary = species_complete_summary[cond]
    
    return species_complete_summary


def _filter_by_contig(species_wgs_summary, max_contig):
    species_wgs_summary = species_wgs_summary.dropna(subset=['contigs']) # must have reported contig count
    species_wgs_summary['contigs'] = species_wgs_summary['contigs'].astype('int') # typecase to numeric
    
    if max_contig:
        species_wgs_summary = species_wgs_summary[species_wgs_summary.contigs <= max_contig]
    else:
        species_wgs_summary = _remove_contig_outliers(species_wgs_summary)

    return species_wgs_summary


def _filter_checkM_contamination(species_wgs_summary, contamination_cutoff):
    species_wgs_summary = species_wgs_summary.dropna(subset=['checkm_contamination']) # must have reported Contamination score
    species_wgs_summary.loc[:,'checkm_contamination'] = species_wgs_summary['checkm_contamination'].astype('float') # typecase to numeric
    
    if contamination_cutoff:
        cond = species_wgs_summary.checkm_contamination < contamination_cutoff
    else:
        kneebow_cutoff = _get_kneebow_cutoff(species_wgs_summary, column='checkm_contamination', curve='elbow')
        cond = species_wgs_summary.checkm_contamination < kneebow_cutoff
        print('CheckM Contamination Cutoff:', kneebow_cutoff)
        
    species_wgs_summary = species_wgs_summary[cond]
    
    return species_wgs_summary


def _filter_checkM_completeness(species_wgs_summary, completeness_cutoff):
    species_wgs_summary = species_wgs_summary.dropna(subset=['checkm_completeness']) # must have reported Completeness score
    species_wgs_summary['checkm_completeness'] = species_wgs_summary['checkm_completeness'].astype('float') # typecase to numeric
    
    if completeness_cutoff:
        cond = species_wgs_summary.checkm_completeness > completeness_cutoff
    else:
        kneebow_cutoff = _get_kneebow_cutoff(species_wgs_summary, column='checkm_completeness', curve='knee')
        cond = species_wgs_summary.checkm_completeness > kneebow_cutoff
        print('CheckM Completeness Cutoff:', kneebow_cutoff)
        
    species_wgs_summary = species_wgs_summary[cond]
    
    return species_wgs_summary

# Helper functions

def append_entry(df_filtration, entry_name, entry_remaining):
    df_temp = pd.DataFrame(index=[entry_name], columns=df_filtration.columns)
    
    df_temp.loc[entry_name, 'initial'] = df_filtration.iloc[-1]['remaining']
    df_temp.loc[entry_name, 'remaining'] = entry_remaining
    df_temp.loc[entry_name, 'num_filtered'] = \
        df_temp.loc[entry_name, 'initial'] \
        - df_temp.loc[entry_name, 'remaining']
    
    df_filtration = pd.concat([df_filtration, df_temp])

    return df_filtration


def _remove_contig_outliers(species_wgs_summary):
    df = species_wgs_summary.copy()
    
    # Step 1: Calculate the IQR
    Q1 = df['contigs'].quantile(0.25)
    Q3 = df['contigs'].quantile(0.75)
    IQR = Q3 - Q1
    
    # Step 2: Find the upper fence
    upper_fence = Q3 + 1.5 * IQR
    print('Congtig upper value:', upper_fence)
    # Step 3: Remove all entries above upper fence limit (clear outliers)
    outlier_cond = species_wgs_summary.contigs <= upper_fence
    species_wgs_summary = species_wgs_summary[outlier_cond]
    
    # Step 4: Remove all entries > 2.5 * median of remaining values
    upper_limit_median = 2.5 * species_wgs_summary.contigs.median()
    outlier_cond2 = species_wgs_summary.contigs <= upper_limit_median
    species_wgs_summary = species_wgs_summary[outlier_cond2]
    
    return species_wgs_summary


def _get_kneebow_cutoff(species_wgs_summary, column, curve):
    df = species_wgs_summary.copy()

    if curve.lower() == 'elbow':
        df = df.sort_values(by=column, ascending=True)
    elif curve.lower() == 'knee':
        df = df.sort_values(by=column, ascending=False)
    else:
        raise ValueError(f'column must be either "elbow" or "knee". {column} was provided instead.')
    
    # set strain index according to ascending/descending values (depending on knee or elbow)
    df = df.reset_index()
    
    # transform input into form necessary for kneebow package
    results_itr = zip(list(df.index), list(df[column]))
    data = list(results_itr)
    
    rotor = Rotor()
    rotor.fit_rotate(data)
    
    if curve.lower() == 'elbow':
        elbow_idx = rotor.get_elbow_index()
        kneebow_cutoff = df[column][elbow_idx]
    elif curve.lower() == 'knee':
        knee_idx = rotor.get_knee_index()
        kneebow_cutoff = df[column][knee_idx]
    
    return kneebow_cutoff



## _INSERT YOUR SPECIES NAME HERE_

In [ ]:
SPECIES_NAME = 'Enterobacter' # Just writing the genus name also works

In [ ]:
summary.genome_name.str.contains(SPECIES_NAME).sum()

In [ ]:
# How many strains of the species/genus are available
species_summary = filter_by_species(summary, SPECIES_NAME)

display(
    species_summary.shape,
    species_summary.head()
)

## Initial Plot (unfiltered data)

In [ ]:
# Initial unfiltered strain plot
h = sns.jointplot(
    data=species_summary,
    x="genome_length",
    y="patric_cds",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
    title='BV-BRC\nstrain type',
)

h.ax_joint.set_xlabel("genome length")
h.ax_joint.set_ylabel("BV-BRC predicted gene count")
plt.show()

In [ ]:
# Find reference strain N50 value from NCBI Genome and multiply by 0.85
# If your species/genus has multiple reference strains, pick the smallest by genome length
# If you are still confused, just send Sidd an email

# Only applies for Complete sequences
species_complete_summary = species_summary[species_summary.genome_status == 'Complete']

fig, ax = plt.subplots()


species_ref_n50 = 4.2e6
min_thresh_n50 = int(0.9 * species_ref_n50)

# Most Complete sequences pass this threshold
sns.histplot(species_complete_summary.contig_n50.dropna().astype('int'), ax=ax)
plt.axvline(x=min_thresh_n50, color='#ff00ff', linestyle='--')
plt.savefig('../images/supplemental/n50.svg', format='svg', dpi=300, bbox_inches='tight')

## Initial Filtration Report

In [ ]:
filtered_species_summary, df_filtration = filter_by_genome_quality(
    species_summary,
    min_thresh_n50=min_thresh_n50,
    max_contig=None,
    contamination_cutoff=None,
    completeness_cutoff=None,
    return_stats=True,
)

display(
    f'Filtered Strains:',
    filtered_species_summary.shape,
    f'------------------------------',
    f'Filtration Report',
    df_filtration
)

In [ ]:
# Same initial plot but with only (first-pass) filtered strains
# For this plot, make sure your WGS sequences form a nice line
# Complete sequences may be all over the place

# For this example, 4Mb looks like a good cutoff to for min genome length
# and 10k looks like a good cutoff to for max predicted genes (though we could go lower)

h = sns.jointplot(
    data=filtered_species_summary,
    x="genome_length",
    y="patric_cds",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
 title='BV-BRC\nstrain type'
)

h.ax_joint.set_xlabel("genome length")
h.ax_joint.set_ylabel("BV-BRC predicted gene count")
plt.show()

# (Optional) Filter by genome length and predicted cds count

## Filter by genome length

In [ ]:
# This step is optional (only needed if you are filtering more strains)
min_genome_length = 4e6
max_predicted_cds = 1e4

# Step 1: Filter strains by min genome length
cond = filtered_species_summary['genome_length'] > min_genome_length
filtered_species_summary = filtered_species_summary[cond]

# Step 2: Calculate min genome length statistics and add to df_filtration
df_temp = pd.DataFrame(index=['min_genome_length'], columns=df_filtration.columns)

df_temp.loc['min_genome_length', 'initial'] = df_filtration.loc['CheckM_completeness_contamination', 'remaining']
df_temp.loc['min_genome_length', 'remaining'] = filtered_species_summary.shape[0]
df_temp.loc['min_genome_length', 'num_filtered'] = \
    df_temp.loc['min_genome_length', 'initial'] \
    - df_temp.loc['min_genome_length', 'remaining']

df_filtration = pd.concat([df_filtration, df_temp])

# Step 3: Filter strains by max predicted cds
cond = filtered_species_summary['patric_cds'] < max_predicted_cds
filtered_species_summary = filtered_species_summary[cond]

# Step 4: Calculate max genome length statistics and add to df_filtration
df_temp = pd.DataFrame(index=['max_predicted_cds'], columns=df_filtration.columns)

df_temp.loc['max_predicted_cds', 'initial'] = df_filtration.loc['min_genome_length', 'remaining']
df_temp.loc['max_predicted_cds', 'remaining'] = filtered_species_summary.shape[0]
df_temp.loc['max_predicted_cds', 'num_filtered'] = \
    df_temp.loc['max_predicted_cds', 'initial'] \
    - df_temp.loc['max_predicted_cds', 'remaining']

df_filtration = pd.concat([df_filtration, df_temp])

df_filtration

In [ ]:
# Second look at filtered plot
h = sns.jointplot(
    data=filtered_species_summary,
    x="genome_length",
    y="patric_cds",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
 title='BV-BRC\nstrain type'
)

h.ax_joint.set_xlabel("genome length")
h.ax_joint.set_ylabel("BV-BRC predicted gene count")
plt.show()

# (Optional) Filter by GC content and Plasmid Count

In [ ]:
# Ensure GC content makes sense
# Remove any big outliers (in this case, anything < 55 and anything > 60)

h = sns.jointplot(
    data=filtered_species_summary,
    x="gc_content",
    y="contigs",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
    title='PATRIC\nstrain type',
    bbox_to_anchor=(1.45,1.4)
)

h.ax_joint.set_xlabel("GC Content")
h.ax_joint.set_ylabel("number of contigs")
plt.savefig('../images/supplemental/gc_content.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ensure Plasmid Count Makes Sense
# Remove any big outliers (in this case, anything > 20)
filtered_species_summary.plasmids = filtered_species_summary.plasmids.fillna(0)
filtered_species_summary.plasmids = filtered_species_summary.plasmids.astype(int)

h = sns.jointplot(
    data=filtered_species_summary,
    x="plasmids",
    y="genome_length",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
    title='PATRIC\nstrain type',
    bbox_to_anchor=(1.45,1.4)
)

h.ax_joint.set_xlabel("Number of Plasmids")
h.ax_joint.set_ylabel("Genome Length")
plt.show()

In [ ]:
# This step is optional (only needed if you are filtering more strains)
gc_content_min = 54
gc_content_max = 57

# Step 1: Filter strains by gc_content
cond1 = filtered_species_summary['gc_content'] > gc_content_min
cond2 = filtered_species_summary['gc_content'] < gc_content_max
filtered_species_summary = filtered_species_summary[cond1 & cond2]

# Step 2: Filter based on plasmid count
# select genomes have too many plasmids, several outliers
cond = filtered_species_summary.plasmids.apply(float) < 20
filtered_species_summary = filtered_species_summary[cond]

# Step 3: Calculate min genome length statistics and add to df_filtration
df_temp = pd.DataFrame(index=['gc_content'], columns=df_filtration.columns)

df_temp.loc['gc_content', 'initial'] = df_filtration.loc['min_genome_length', 'remaining']
df_temp.loc['gc_content', 'remaining'] = filtered_species_summary.shape[0]
df_temp.loc['gc_content', 'num_filtered'] = \
    df_temp.loc['gc_content', 'initial'] \
    - df_temp.loc['gc_content', 'remaining']

df_filtration = pd.concat([df_filtration, df_temp])

df_filtration

In [ ]:
# Second look at filtered plot
h = sns.jointplot(
    data=filtered_species_summary,
    x="gc_content",
    y="contigs",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
    title='PATRIC\nstrain type',
    bbox_to_anchor=(1.45,1.4)
)

h.ax_joint.set_xlabel("GC Content")
h.ax_joint.set_ylabel("number of contigs")
plt.show()

In [ ]:
# Num of (first-pass) filtered strains (for NMF)
cond = filtered_species_summary.genome_status == 'Complete'
filtered_species_summary[cond].shape[0]

In [ ]:
# Make sure contig distribution looks good (little to no outliers)
# For this example, it looks okay (only 11 outliers with all of them being close to the upper fence)
px.box(filtered_species_summary.contigs)

In [ ]:
h = sns.jointplot(
    data=filtered_species_summary,
    x="contig_n50",
    y="contigs",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
    title='PATRIC\nstrain type',
    bbox_to_anchor=(1.45,1.4)
)

h.ax_joint.set_xlabel("Contig N50")
h.ax_joint.set_ylabel("number of contigs")
plt.show()

In [ ]:
display(
    filtered_species_summary.contig_n50.median(),
    sns.histplot(data=filtered_species_summary, x='contig_n50')
)

In [ ]:
display(
    filtered_species_summary.contig_l50.median(),
    sns.histplot(data=filtered_species_summary, x='contig_l50')
)

In [ ]:
h = sns.jointplot(
    data=filtered_species_summary,
    x="contig_n50",
    y="contig_l50",
    hue="genome_status",
    alpha=0.75,
    height=4
)

h.ax_joint.legend(
    title='BV-BRC\nstrain type',
    bbox_to_anchor=(1.45,1.4)
)

h.ax_joint.set_xlabel("Contig N50")
h.ax_joint.set_ylabel("Contig L50")
plt.show()

# Save (first-pass) filtered Genome Summary & Genome Metadata Files

In [ ]:
PAN_PHYLON_DIR = '../../data'
# NUM_WORDS = len(SPECIES_NAME.split(' '))

filepath = os.path.join(PAN_PHYLON_DIR)

# filepath = None
# if NUM_WORDS > 1: # Most species
#     filepath = os.path.join(PAN_PHYLON_DIR, SPECIES_NAME.split(' ')[0][0] + '_' + SPECIES_NAME.split(' ')[1])
# elif NUM_WORDS == 1: # In case you are doing genus-level analysis
#     filepath = os.path.join(PAN_PHYLON_DIR, SPECIES_NAME)
# else:
#     raise ValueError(f'SPECIES_NAME is must contain 1 word (for genus) or 2 (for species). {NUM_WORDS} words found instead.')

filepath = os.path.join(filepath, 'metadata')
filepath = os.path.join(filepath, 'filtered_species_summary.csv')
filtered_species_summary.to_csv(filepath)

In [ ]:
filtered_species_metadata = metadata.loc[filtered_species_summary.index]

filepath = os.path.join(filepath.split('filtered_species_summary.csv')[0], 'filtered_species_metadata.csv')
filtered_species_metadata.to_csv(filepath)